# 27 - The green misses: star wings in the noise, or the model?

**Purpose.** Re-measure the noise of the four sky-pair cells with the stars masked out, and
publish the four pairs again **only if the numbers change**. Notebook `25` found the model passes
MISSION's test in green on three pairs of four, but misses all four in the same direction - by
7.9 points on average, 1.8 to 4.2 times the repeatability. This notebook asks whether that miss
is in the model or in the measurement.

**Why stars.** Section 1 below lays out the evidence, and it is the reason this notebook exists:

- The two cheap suspects `25` named are ruled out by arithmetic on published numbers alone. The
  dark bound is 0.0005 e-/px/s against a sky near 1.9. The object's own light is about 6% of the
  sky, and putting it into the noise moves no prediction by more than 2 points.
- Chaining the four pairs gives each cell's shortfall against the model, cell A as the zero. The
  shortfalls line up with how many pixels clip in each cell: A least, then B, C, and D worst by far.
  Blue clips less than green, and blue misses less.
- `stats.diff_sigma` already clips star *cores* at 5 sigma. What it cannot reach is the *wings* of a
  star whose profile changed between the two half-stacks - seeing, or a clipped core that the
  rejection treats differently. Those residues sit below the clip and widen the spread, and the
  brighter the stars are in counts, the wider. That would inflate the noise most in D.

**The test, fixed before any number is seen.** One mask for every cell, so every cell loses the
same patch of sky: stars found with `spatial.star_mask` on each cell's full stack, per plane, and
the union taken over all sixteen. `stats.diff_sigma` then leaves out every 4x4 block that touches
the mask. Stars are anything 5 sigma above the local background, and **`tight`** grows each one
by 3 plane pixels; it is the one the verdict uses. The check is a ladder under it - grown by
**0, 1 and 2** pixels - because if star wings are the cause, the noise falls as the mask grows
and then levels off before 3.

*The check was first set as a second mask grown by 6. It was replaced before any noise was
measured, because this is a Milky Way field: grown by 6, the union of sixteen masks covers 90%
of the signal ROI and leaves 756 blocks, too few for a spread (section 4 prints the coverage).
`tight` itself leaves about 2400.*

- **Stars are the cause** if the masked noise falls most in D and least in A, and the green
  misses shrink towards the predictions by more than their repeatability.
- **Stars are not the cause** if the masked noise falls by the same fraction in every cell. Then
  the miss survives, it is a model question, and MISSION's last untested assumption - a noise term
  that does not shrink with `t` - is back in front.

**Publishing rule, fixed here.** If any green pair's measured separation moves by more than its
own repeatability under `tight`, this notebook writes `results/masked_pairs.csv` and
`results/mask_constants.json`, with new names that sit beside `25`'s instead of overwriting them.
If none moves that far, it writes nothing, and the answer goes in the decision log.

**What is held fixed, so that only the noise can move.** The predictions are `17`'s, unchanged.
The signal is `25`'s: `stats.extended_signal`, medians, which are already robust to stars and are
not masked here. Registration, the reference, the frames, the interleave, the rejection settings,
the ROIs and the 4x4 scale are all `25`'s. Section 4 re-measures the unmasked pairs and **stops
the notebook** if they do not reproduce `stack_pairs.csv`, because otherwise a change could come
from the re-run and not from the mask.

**What it is not for.** Not the combination-efficiency ladder: only the stacks the SNR needs are
built. Not new predictions. Not a star-by-star clipping count (the star-colour half of the Pareto
curve), even though a star mask is on the way to one: that needs per-star photometry, and it has
its own purpose to agree. Not an explainer: if this notebook publishes, the reading-back one
follows it.

**It assumes** `25_contract3_stacks.ipynb` for the pipeline and the SNR estimator, and
`00_statistics.ipynb` for why a spread over thousands of blocks is sharp.

In [ ]:
import functools
import json
import pathlib
import sys
import time

import numpy as np
import pandas as pd

sys.path.insert(0, str(pathlib.Path.cwd().parent))
from astropix import model as M, pixinsight as PI, spatial, stats

ROOT = pathlib.Path.cwd().parent
RESULTS = ROOT / "results"
DATA = ROOT / "data" / "session06"
# Disposable, on the NAS (CLAUDE.md), and a folder of its own so that 25's
# layout is not assumed.  Cells skip work whose result file is already here.
TEMP = pathlib.Path(r"Z:\pix\_astro\astropix\temp\contract3\mask")
for sub in ("split", "reg", "ref", "stack"):
    (TEMP / sub).mkdir(parents=True, exist_ok=True)

sky = json.loads((RESULTS / "sky_constants.json").read_text())
SIGNAL_ROI = spatial.plane_roi(sky["sky_roi"]["value"]["signal_roi"])
SKY_ROI = spatial.plane_roi(sky["sky_roi"]["value"]["roi"])
T_DEAD = sky["t_dead"]["value"]
K = stats.NOISE_BIN

CELLS = {"A": (50, 30.0), "B": (50, 120.0), "C": (200, 30.0), "D": (200, 120.0)}
PLANES = spatial.PLANES
# 25's winsorized arm, unchanged
SETTINGS = {"combination": "average", "normalization": "additive", "weight_mode": "dont_care",
            "rejection": "winsorized", "sigma_low": 4.0, "sigma_high": 3.0}
MASKS = {f"grow{g}": {"nsigma": 5.0, "grow": g} for g in (0, 1, 2)}
MASKS["tight"] = {"nsigma": 5.0, "grow": 3}
VERDICT_MASK = "tight"

PAIRS_CSV = RESULTS / "masked_pairs.csv"
CONSTANTS = RESULTS / "mask_constants.json"
published = pd.read_csv(RESULTS / "stack_pairs.csv")

probe = PI.run(PI.scripts_dir() / "probe.js", {"an_int": 42})
assert probe["ok"], probe.get("error")
print(f"PixInsight {probe['core']['version']} build {probe['core']['build']}")
print(f"signal ROI {SIGNAL_ROI}, sky ROI {SKY_ROI} (plane pixels), t_dead {T_DEAD:.2f} s")

## 1. Why stars: the two dead suspects, and the shortfall per cell

Everything here is arithmetic on published files; nothing is read from a frame.

**The two suspects `25` named.** Dark current is `17`'s -10 C bound, an upper bound on a -20 C
night. The object's flux is `(signal median - sky median) * g / t` per frame, from
`sky_frames.csv`, with `17`'s own cold `g`. Each is put into `17`'s prediction to see how far it
moves.

In [ ]:
cold = json.loads((RESULTS / "cold_constants.json").read_text())
G = {int(k): v for k, v in cold["system_gain_cold_per_plane"]["value"].items()}
R_E = {int(k): v for k, v in cold["read_noise_cold_e"]["value"].items()}
D_BOUND = 0.000534                  # e-/px/s, 17's bound: -10 C, so high for a -20 C night
PLANE_SET = {"green": ["G1", "G2"], "B": ["B"]}

frames = pd.read_csv(RESULTS / "sky_frames.csv")
frames = frames[~frames.flipped & ~frames.rejected]
flux = {}
for cell, (g, t) in CELLS.items():
    f = frames[frames.cell == cell]
    for plane, ps in PLANE_SET.items():
        flux[cell, plane] = {
            "sky": np.mean([f[f"F_sky_{p}"].mean() for p in ps]),
            "obj": np.mean([((f[f"sig_med_{p}"] - f[f"sky_med_{p}"]) * G[g][p] / t).mean()
                            for p in ps])}


def rel_snr(cell, plane, f_obj=0.0, dark=D_BOUND):
    g, t = CELLS[cell]
    return np.sqrt(t / (t + T_DEAD)) / np.sqrt(flux[cell, plane]["sky"] + f_obj + dark
                                               + R_E[g] ** 2 / t)


rows = []
for r in published.itertuples():
    a, b = r.pair.split("->")
    def pct(**kw):
        return 100 * (rel_snr(b, r.plane, **kw) / rel_snr(a, r.plane, **kw) - 1)
    rows.append({"pair": r.pair, "plane": r.plane, "predicted_17": r.predicted_pct,
                 "no_dark": pct(dark=0.0),
                 "with_F_obj": pct(f_obj=np.mean([flux[a, r.plane]["obj"],
                                                  flux[b, r.plane]["obj"]])),
                 "measured": r.measured_pct, "repeatability": r.repeatability_pct})
suspects = pd.DataFrame(rows)
print(suspects.round(2).to_string(index=False))
print("\nF_obj / F_sky per cell, green:",
      {c: round(flux[c, "green"]["obj"] / flux[c, "green"]["sky"], 3) for c in CELLS})
print(f"largest move from dropping the dark term: "
      f"{(suspects.no_dark - suspects.predicted_17).abs().max():.3f} points")
print(f"largest move from adding F_obj to the noise: "
      f"{(suspects.with_F_obj - suspects.predicted_17).abs().max():.2f} points")

**The shortfall per cell.** Each pair's measured ratio over its predicted ratio says how far the
second cell fell short *relative to* the first. Set A to 1 and chain A->B, A->C and C->D; B->D is
then a free check, because it is implied by the other three. Beside it, `17`'s clipped-pixel
fraction in the signal ROI, the plainest count of how bright the stars are in counts.

In [ ]:
clip = sky["clipped_fraction"]["value"]
ratio = {(r.pair, r.plane): (1 + r.measured_pct / 100) / (1 + r.predicted_pct / 100)
         for r in published.itertuples()}
short = []
for plane, ps in PLANE_SET.items():
    s = {"A": 1.0}
    s["B"] = ratio["A->B", plane]
    s["C"] = ratio["A->C", plane]
    s["D"] = s["C"] * ratio["C->D", plane]
    check = s["D"] / s["B"] / ratio["B->D", plane]
    for c in CELLS:
        short.append({"plane": plane, "cell": c, "shortfall_pct": 100 * (s[c] - 1),
                      "clipped_pct": 100 * np.mean([clip[c][p] for p in ps])})
    print(f"{plane}: B->D implied by the other three over B->D measured = {check:.4f}")
short = pd.DataFrame(short)
print(short.pivot(index="cell", columns="plane").round(3).to_string())

## 2. The frames, the reference, and registration

`25`'s rules, run again: the good frames are `17`'s, the reference is the cell D frame nearest the
middle of the post-flip night, and it is left out of every stack. About twelve minutes a cell.

In [ ]:
frames = pd.read_csv(RESULTS / "sky_frames.csv", parse_dates=["t"])
good = frames[~frames.flipped & ~frames.rejected].sort_values("t").reset_index(drop=True)
good["stem"] = [f"{c}_{t:%H%M%S}" for c, t in zip(good.cell, good.t)]
assert good.stem.is_unique

mid = good.t.min() + (good.t.max() - good.t.min()) / 2
d = good[good.cell == "D"]
REF = d.loc[(d.t - mid).abs().idxmin()]
stackable = good[good.stem != REF.stem]
print(f"reference: {REF.file}")
print(stackable.groupby("cell").size().rename("frames").to_frame().T.to_string())

reg = {}
for cell in CELLS:
    out = TEMP / f"register_{cell}.json"
    if not out.exists():
        rows = stackable[stackable.cell == cell]
        job = {"reference": PI.pi_path(DATA / REF.file), "outdir": PI.pi_path(TEMP),
               "interpolation": "lanczos3",
               "frames": [{"path": PI.pi_path(DATA / f), "stem": s}
                          for f, s in zip(rows.file, rows.stem)]}
        t0 = time.time()
        r = PI.run(PI.scripts_dir() / "register.js", job, timeout=3600)
        assert r["ok"], r.get("error")
        r.pop("log", None)
        out.write_text(json.dumps(r, indent=1))
        print(f"cell {cell}: {len(rows)} frames registered in {time.time() - t0:.0f} s")
    reg[cell] = json.loads(out.read_text())

qual = pd.DataFrame([{"cell": c, "stem": f["stem"], "plane": p, "ok": v["ok"],
                      "path": v.get("output", "")}
                     for c, r in reg.items() for f in r["frames"] for p, v in f["planes"].items()])
assert qual.ok.all(), f"{(~qual.ok).sum()} frame-planes failed to register"
path_of = qual.set_index(["stem", "plane"]).path
print(f"all {len(qual)} frame-planes registered")

## 3. The stacks the SNR needs

`25`'s `hA`/`hB` halves, `q0`...`q3` quarters and `full`, winsorized, and nothing else.

In [ ]:
def plan(rows):
    ev, od = list(rows.stem[0::2]), list(rows.stem[1::2])
    m, m4 = min(len(ev), len(od)), len(rows) // 4
    return ({"hA": ev[:m], "hB": od[:m]}
            | {f"q{i}": list(rows.stem[i::4])[:m4] for i in range(4)}
            | {"full": list(rows.stem)})


stacks = {}
for cell in CELLS:
    out = TEMP / f"integrate_{cell}.json"
    rows = stackable[stackable.cell == cell].reset_index(drop=True)
    runs = [{"label": f"{cell}|{p}|{label}",
             "output": PI.pi_path(TEMP / "stack" / f"{cell}_{p}_{label}.fits"),
             "frames": [PI.pi_path(path_of[s, p]) for s in stems], **SETTINGS}
            for p in PLANES for label, stems in plan(rows).items()]
    if not out.exists():
        t0 = time.time()
        r = PI.run(PI.scripts_dir() / "integrate.js", {"runs": runs}, timeout=5400)
        assert r["ok"], r.get("error")
        r.pop("log", None)
        out.write_text(json.dumps(r, indent=1))
        print(f"cell {cell}: {len(runs)} integrations in {time.time() - t0:.0f} s")
    r = json.loads(out.read_text())
    assert all(x["ok"] for x in r["runs"]), [x["label"] for x in r["runs"] if not x["ok"]]
    stacks |= {x["label"]: {"n": x["n"], "path": x["output"]} for x in r["runs"]}
print(f"{len(stacks)} stacks")


@functools.cache
def boxes(cell, p, label):
    """The signal and sky boxes of one stack, read from the NAS once."""
    s = PI.read_adc(stacks[f"{cell}|{p}|{label}"]["path"])
    return spatial.cut(s, SIGNAL_ROI).copy(), spatial.cut(s, SKY_ROI).copy()

## 4. The masks

Stars are found on each cell's full stack, per plane, inside the signal ROI, and the sixteen masks
are OR-ed into one. The same pixels are then left out of every cell and every plane. That costs the
cells with fewer visible stars some sky they did not need to lose. It buys the thing a ranking
needs: every cell measured on the same patch.

In [ ]:
full = {(c, p): boxes(c, p, "full")[0] for c in CELLS for p in PLANES}
masks = {"none": None}
for name, kw in MASKS.items():
    masks[name] = np.logical_or.reduce([spatial.star_mask(a, **kw) for a in full.values()])
    blocks = stats.bin_mean(masks[name], K) > 0
    print(f"{name:6s} {100 * masks[name].mean():5.1f}% of pixels, "
          f"{100 * blocks.mean():5.1f}% of {K}x{K} blocks left out, "
          f"{int((~blocks).sum())} blocks kept")

## 5. Noise and SNR, three ways

`25`'s estimator, cell for cell: the full stack's noise from the half-split scaled by frame count,
the repeatability the larger of the half-split and the quarter scatter. The only thing that changes
between the three runs is the mask handed to `stats.diff_sigma`.

**The gate.** With no mask this must give back `stack_pairs.csv`. It is checked against half of
each pair's repeatability, because a re-run that moves by more than that could explain any change
the mask makes.

In [ ]:
def snr_table(mask):
    def sig(cell, p, a, b):
        return stats.diff_sigma(boxes(cell, p, a)[0], boxes(cell, p, b)[0], K, mask=mask)

    def snr_of(cell, p, label, noise):
        return stats.extended_signal(*boxes(cell, p, label)) / noise

    out = []
    for cell in CELLS:
        n_full = stacks[f"{cell}|R|full"]["n"]
        m, m4 = stacks[f"{cell}|R|hA"]["n"], stacks[f"{cell}|R|q0"]["n"]
        for p in PLANES:
            s_half = sig(cell, p, "hA", "hB")
            s_qa, s_qb = sig(cell, p, "q0", "q2"), sig(cell, p, "q1", "q3")
            ha = snr_of(cell, p, "hA", s_qa * np.sqrt(m4 / m))
            hb = snr_of(cell, p, "hB", s_qb * np.sqrt(m4 / m))
            q = [snr_of(cell, p, f"q{i}", (s_qa, s_qb, s_qa, s_qb)[i]) for i in range(4)]
            out.append({"cell": cell, "plane": p, "n": n_full,
                        "noise": s_half * np.sqrt(m / n_full),
                        "snr": snr_of(cell, p, "full", s_half * np.sqrt(m / n_full)),
                        "rep_pct": max(100 * abs(ha / hb - 1) / 2,
                                       100 * np.std(q, ddof=1) / np.mean(q) / 2)})
    return pd.DataFrame(out)


def pairs_of(snr):
    def cell_snr(cell, plane):
        rows = snr[snr.cell == cell].set_index("plane").loc[PLANE_SET[plane]]
        rep = float(np.sqrt((rows.rep_pct ** 2).mean()))
        return M.snr_per_root_night(rows.snr.mean(), int(rows.n.iloc[0]),
                                    CELLS[cell][1], T_DEAD), rep

    out = []
    for r in published.itertuples():
        a, b = r.pair.split("->")
        (s1, rep1), (s2, rep2) = cell_snr(a, r.plane), cell_snr(b, r.plane)
        measured, rep = 100 * (s2 / s1 - 1), float(np.hypot(rep1, rep2))
        out.append({"pair": r.pair, "why": r.why, "plane": r.plane,
                    "predicted_pct": r.predicted_pct, "measured_pct": measured,
                    "repeatability_pct": rep,
                    "verdict": M.verdict(r.predicted_pct, measured, rep)})
    return pd.DataFrame(out)


snr = {name: snr_table(m) for name, m in masks.items()}
pairs = {name: pairs_of(s) for name, s in snr.items()}

gate = pairs["none"].merge(published, on=["pair", "plane"], suffixes=("", "_25"))
gate["moved"] = gate.measured_pct - gate.measured_pct_25
print(gate[["pair", "plane", "measured_pct_25", "measured_pct", "moved",
            "repeatability_pct_25"]].round(3).to_string(index=False))
assert (gate.moved.abs() <= gate.repeatability_pct_25 / 2).all(), (
    "the unmasked re-run does not reproduce 25; stop -- a change below could be the re-run")
print("\nthe re-run reproduces 25 without a mask")

## 6. What the mask did to the noise, cell by cell

The test from the purpose cell. Stars predict the noise falls most in D and least in A; a model
problem predicts it falls by the same fraction everywhere.

In [ ]:
drop = []
for name in MASKS:
    for plane, ps in PLANE_SET.items():
        for c in CELLS:
            n0 = snr["none"].set_index(["cell", "plane"]).loc[[(c, p) for p in ps], "noise"].mean()
            n1 = snr[name].set_index(["cell", "plane"]).loc[[(c, p) for p in ps], "noise"].mean()
            drop.append({"mask": name, "plane": plane, "cell": c,
                         "noise_change_pct": 100 * (n1 / n0 - 1)})
drop = pd.DataFrame(drop).merge(short, on=["plane", "cell"])
print(drop.pivot_table(index=["mask", "cell"], columns="plane",
                       values=["noise_change_pct", "shortfall_pct"]).round(2).to_string())

## 7. The four pairs, masked

In [ ]:
side = pairs["none"][["pair", "plane", "predicted_pct", "measured_pct", "repeatability_pct",
                      "verdict"]].copy()
for name in MASKS:
    side = side.merge(pairs[name][["pair", "plane", "measured_pct", "repeatability_pct",
                                   "verdict"]], on=["pair", "plane"], suffixes=("", f"_{name}"))
print(side.round(2).to_string(index=False))

v = pairs[VERDICT_MASK].merge(pairs["none"], on=["pair", "plane"], suffixes=("", "_none"))
v["moved"] = v.measured_pct - v.measured_pct_none
green = v[v.plane == "green"]
changed = bool((green.moved.abs() > green.repeatability_pct_none).any())
miss = (green.measured_pct - green.predicted_pct)
miss0 = (green.measured_pct_none - green.predicted_pct)
print(f"\ngreen mean miss: {miss0.mean():+.2f} points unmasked, {miss.mean():+.2f} with "
      f"'{VERDICT_MASK}'")
print(f"green verdicts with '{VERDICT_MASK}': {green.verdict.value_counts().to_dict()}")
print("the publishing rule:", "a green pair moved by more than its repeatability -- PUBLISH"
      if changed else "no green pair moved by more than its repeatability -- nothing is written")

## 8. Publish, if the rule says so

`masked_pairs.csv` holds every pair under every mask, with the unmasked re-run beside it.
`mask_constants.json` holds the verdict under `tight` as `ranked_pairs_masked` - a new name, so it
sits beside `25`'s `ranked_pairs` and a model loading both files meets no conflict - and the mask
itself, as a constant with the settings that made it.

In [ ]:
if changed:
    measured_on = str(good.t.min().date())

    def constant(value, unit, uncertainty, note):
        return {"value": value, "unit": unit, "uncertainty": uncertainty,
                "source_frames": int(len(stackable)), "measured_on": measured_on,
                "notebook": "27_star_mask.ipynb", "note": note}

    out = pd.concat([p.assign(mask=name) for name, p in pairs.items()], ignore_index=True)
    out.round(6).to_csv(PAIRS_CSV, index=False)
    constants = {
        "ranked_pairs_masked": constant(
            {f"{r.pair} {r.plane}": r.verdict for r in pairs[VERDICT_MASK].itertuples()},
            "verdict per pair and plane: pass, fail or tie", None,
            f"MISSION's definition of done, applied by model.verdict, with the noise measured "
            f"under the '{VERDICT_MASK}' star mask.  Predictions are 17's and the signal is 25's, "
            f"both unchanged; only the noise moved"),
        "star_mask": constant(
            {name: {**kw, "pixel_pct": round(100 * float(masks[name].mean()), 3),
                    "block_pct": round(100 * float((stats.bin_mean(masks[name], K) > 0).mean()), 3)}
             for name, kw in MASKS.items()},
            "spatial.star_mask settings, and the share of the signal ROI they leave out", None,
            f"one mask for every cell and plane: the union of spatial.star_mask on each cell's "
            f"full winsorized stack, per plane, in the signal ROI {SIGNAL_ROI}.  diff_sigma drops "
            f"every {K}x{K} block that touches it"),
    }
    with open(CONSTANTS, "w") as f:
        json.dump(constants, f, indent=2)
    M.Constants.load(CONSTANTS)       # the provenance gate
    print("wrote", PAIRS_CSV.name, CONSTANTS.name)
else:
    print("nothing written: the mask did not move the pairs")

## 9. Clean up

The registered planes and stacks are disposable and several GB.

In [ ]:
import shutil
shutil.rmtree(TEMP, ignore_errors=True)
print("removed", TEMP)